In [0]:
from pyspark.sql.functions import sum, col, round as spark_round, abs as spark_abs

In [0]:
df = spark.read.json("/Volumes/mining/ground_truth/truth").filter(col("shift_id").isNotNull())

In [0]:
df.show(5)

In [0]:
df.printSchema()

In [0]:
truth_totals = df.groupBy("shift_id","truck_id").agg(spark_round(sum("true_tonnage"),1).alias("true_total"))

In [0]:
truth_totals.show()

In [0]:
gold = spark.read.table("mining.gold.production_reconciliation")

In [0]:
comparison = gold.join(truth_totals, on=["shift_id","truck_id"], how="inner")

In [0]:
comparison = comparison.withColumn("difference", spark_abs(col("total_tonnage") - col("true_total")))

In [0]:
comparison = comparison.withColumn("difference", spark_abs(col("total_tonnage") - col("true_total")))
comparison.show()

In [0]:
comparison.show()

In [0]:
mismatches = comparison.filter(col("difference") > 0.1)
print("Mismatched trucks:", mismatches.count())

In [0]:
# quarantine count — how many caught
spark.read.table("mining.silver.quarantine").count()

In [0]:
comparison.orderBy(col("difference").desc()).show()

In [0]:
mismatches = comparison.filter(col("difference") > 0.1)
print("Truck-shifts with mismatch:", mismatches.count())
comparison.orderBy(col("difference").desc()).show(10)

In [0]:
spark.read.table("mining.silver.haul_events").groupBy("truck_id").count().orderBy("truck_id").show()

In [0]:
print("=== SCENARIO 2: DUPLICATES — DEDUP ON (after) ===")
print("Bronze rows:", spark.read.table("mining.bronze.haul_events").count())
print("Silver rows:", spark.read.table("mining.silver.haul_events").count())
mismatches = comparison.filter(col("difference") > 0.1)
print("Truck-shifts with mismatch:", mismatches.count())
comparison.orderBy(col("difference").desc()).show(10)

In [0]:
spark.read.table("mining.silver.quarantine").groupBy("shift_id","truck_id").count().orderBy("count", ascending=False).show(10)

In [0]:
print("=== SCENARIO 3B: RENAME — fix ON (after) ===")
spark.read.table("mining.silver.haul_events").groupBy("truck_id").count().orderBy("truck_id").show()

In [0]:
print("=== SCENARIO 1: IMPOSSIBLE TONNAGE — caught in quarantine ===")
print("Quarantined rows:", spark.read.table("mining.silver.quarantine").count())
spark.read.table("mining.silver.quarantine").select("cycle_id","model","reported_tonnage").show(15)